# Ride Smart

O projeto RideSmart aborda o cenário em que um usuário aceita caminhar uma
distância máxima X a partir de uma origem A até um ponto de
embarque otimizado P, visando mitigar congestionamentos,
contornar barreiras geométricas ou acessar vias de tráfego
rápido, reduzindo o tempo ou a distância total de deslocamento
até o destino final B.

### Integrantes

- Henrique Eduardo Costa Da Silva
- João Victor Moura Lucas da Silva
- Murilo de Lima Barros
- Ramon Vinícius Ferreira de Souza

### Preparação do Ambiente

In [ ]:
# Caso queira instalar as dependências diretamente no kernel, descomente a linha abaixo e execute o código
!pip install pandas geopandas osmnx matplotlib scikit-learn folium --quiet

In [ ]:
# Importando as bibliotecas necessárias
import os
import sys
import time
import argparse
import pandas as pd
import osmnx as ox

In [ ]:
# Ajusta o caminho do projeto para permitir importações do src/
CURRENT_DIR = os.path.abspath(os.getcwd())
SRC_PATH = os.path.join(CURRENT_DIR, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

In [ ]:
# Importa os módulos do RideSmart
from main import load_graph, build_adjacency_from_graph, generate_random_points, create_solver, select_best_boarding_node
from visualization import animate_algorithm, create_folium_map

# Dicionário global para armazenar métricas para comparação final
metrics_summary = {}

### Carregando o grafo e preparando as estruturas
Baixa o grafo OSMnx e cria a lista de adjacência para a cidade de Natal, RN. No exemplo, utilizando o algoritmo de Dijkstra com Heap.

In [ ]:
# Definição dos parâmetros base (mude o tráfego ou coordenadas aqui se desejar)
args = argparse.Namespace(
    place='Natal, Brazil',
    graphml=None,
    origin_lat=-5.835069,   # UFRN, Natal
    origin_lon=-35.2113248,
    dest_lat=-5.81116,      # Midway Mall, Natal
    dest_lon=-35.2063,
    walk_radius=200.0,
    candidates=20,
    min_angle=0.0,
    seed=42,
    traffic='off'          # Opções: 'off', 'normal', 'peak'
)

In [ ]:
print('Carregando o grafo (isso pode levar alguns segundos)...')
G = load_graph(args)
G = ox.distance.add_edge_lengths(G)

# Monta a estrutura de adjacência
adjacency, node_to_idx, idx_to_node, coords = build_adjacency_from_graph(G, weight_attr='length')
print(f'Grafo carregado com sucesso! Total de nós: {len(adjacency)}')

### Definindo a Origem, Destino e Candidatos de Embarque

In [ ]:
# Encontra os nós mais próximos no grafo do OpenStreetMap
origin_node = ox.distance.nearest_nodes(G, X=args.origin_lon, Y=args.origin_lat)
dest_node = ox.distance.nearest_nodes(G, X=args.dest_lon, Y=args.dest_lat)
dest_idx = node_to_idx[dest_node]

In [ ]:
# Gera os pontos de caminhada máximos (candidatos a embarque)
candidates = generate_random_points(
    args.origin_lat, args.origin_lon, args.walk_radius, args.candidates, args.min_angle, G, seed=args.seed
)
candidate_nodes = list(dict.fromkeys(candidates))
candidate_indices = [node_to_idx[n] for n in candidate_nodes]

print(f'Nós candidatos gerados na área de caminhada (raio {args.walk_radius}m): {len(candidate_nodes)}')

### Avaliando os Algoritmos

In [ ]:
def run_and_eval_algorithm(algo_name):
    print(f"=== Executando {algo_name.upper()} ===")
    
    # 1. Inicializa o Solver correspondente
    solver = create_solver(algo_name, adjacency, coords)
    
    # 2. Mede o tempo de execução do cálculo e seleção da rota
    start_time = time.perf_counter()
    best_node, best_weight, best_path = select_best_boarding_node(candidate_nodes, node_to_idx, solver, dest_idx)
    end_time = time.perf_counter()
    
    execution_time_ms = (end_time - start_time) * 1000
    
    # 3. Salva os resultados no dicionário global
    metrics_summary[algo_name] = {
        'Tempo de Execução (ms)': execution_time_ms,
        'Melhor Nó': best_node,
        'Custo da Rota (Metros/Tempo)': best_weight,
        'Nós no Caminho': len(best_path) if best_path else 0
    }
    
    # 4. Exibe métricas básicas na célula
    print(f"Tempo de execução: {execution_time_ms:.2f} ms")
    print(f"Melhor nó encontrado: {best_node}")
    print(f"Custo total estimado até o destino: {best_weight:.2f}")
    
    # 5. Gera a animação em HTML
    html_animation = animate_algorithm(G, solver, origin_node, dest_node, dest_idx, candidate_indices, idx_to_node, args.walk_radius)
    
    return html_animation, best_path, best_node, best_weight, best_path

#### 1 - Dijkstra Simples

In [ ]:
html, path = run_and_eval_algorithm('dijkstra')
display(html)

#### 2 - Dikstra com Heap

In [ ]:
html, path_heap = run_and_eval_algorithm('dijkstra_heap')
display(html)

#### 3 - A* (A Star)

In [ ]:
html, path_astar, best_node, best_weight, best_path = run_and_eval_algorithm('astar')
display(html)

#### 4 - SPFA

In [ ]:
html, path_spfa = run_and_eval_algorithm('spfa')
display(html)

### Comparando os Tempos

In [ ]:
df_metrics = pd.DataFrame.from_dict(metrics_summary, orient='index')
# Ordena pelo tempo de execução para ver qual foi o mais rápido
df_metrics = df_metrics.sort_values(by='Tempo de Execução (ms)')
df_metrics

### Mapa com Melhor Caminho
Desenha o caminho no mapa (A*).

In [ ]:
# Gera e salva mapa Folium com o melhor caminho
folium_out = os.path.join(CURRENT_DIR, 'data', 'mapa_interativo.html')
os.makedirs(os.path.dirname(folium_out), exist_ok=True)
create_folium_map(G, origin_node, dest_node, candidate_nodes, best_path, idx_to_node, output_path=folium_out)
print('Mapa interativo salvo em', folium_out)

### Discussões

1 - Como o problema foi modelado como grafo?

O problema foi modelado como um grafo direcionado e ponderado $G = (V, E)$, utilizando dados reais da malha urbana de Natal extraídos através da biblioteca OSMnx.

2 - O que representam os nós e as arestas?
- Nós ($V$): Representam os cruzamentos (interseções viárias) e fins de rua, contendo as coordenadas geográficas (latitude e longitude).
- Arestas ($E$): Representam os segmentos de rua transitáveis, com sentido definido (respeitando a mão das vias).

3 - Quais pesos foram usados?

O peso variou conforme o cenário configurado (--traffic):

- length: Distância física real da rua em metros.

- travel_time: Tempo de viagem ideal baseado no limite de velocidade da via.

- travel_time_with_traffic: Tempo de viagem acrescido de um atraso aleatório (distribuição exponencial) para simular trânsito comum ou horário de pico (peak), penalizando principalmente as vias expressas.

4 - Como o trânsito sintético alterou as rotas?

No cenário ideal (sem trânsito), as rotas convergiram para avenidas principais (como a BR-101 e Salgado Filho). Com o trânsito ativado, as vias principais tornaram-se lentas, forçando os algoritmos a desviarem o trajeto por ruas secundárias e bairros adjacentes.

5 - Caminhar alguns metros melhorou a solução?

Sim. Na maioria das vezes, permitiu que o passageiro andasse até uma rua paralela ou mudasse o sentido de uma avenida, fazendo a viagem iniciar já fora de gargalos e evitando retornos demorados.

6 - Em quais casos caminhar atrapalhou?

Atrapalhou quando o destino final ficava em uma direção geometricamente oposta ao ponto de embarque candidato (fazendo o tempo gasto a pé não compensar o ganho do carro) ou quando a malha secundária ao redor também estava completamente congestionada.

7 - A menor distância foi também a rota mais rápida?

Não necessariamente. Especialmente sob tráfego, a rota com menor distância física tendeu a ser mais demorada por cruzar pontos saturados. Rotas mais rápidas muitas vezes exigiram caminhos fisicamente mais longos.

8 - O A* expandiu menos nós que o Dijkstra?Sim. O algoritmo $A^*$ utiliza uma busca guiada por heurísticas geográficas (como a distância de Haversine em linha reta até o destino), focando na direção correta e expandindo substancialmente menos nós que a busca radial/uniforme do Dijkstra.

9 - O Dijkstra com Heap foi mais eficiente que o Dijkstra simples?

Sim, drasticamente. Enquanto o Dijkstra simples levou cerca de segundos varrendo os nós de forma linear, a versão com Heap (fila de prioridades) resolveu o problema em apenas algumas dezenas de milissegundos.

10 - O algoritmo da literatura trouxe algum ganho?

O algoritmo adaptado da literatura (SPFA) foi satisfatório e encontrou o caminho ideal em algumas centenas de milissegundos, mostrando-se muito melhor que o Dijkstra Simples. Contudo, ele não trouxe ganho sobre o $A^*$ e o Dijkstra com Heap, que foram significativamente mais rápidos.

11 - Quais limitações existem na modelagem proposta?

O modelo gerou trânsito de forma matemática (sintética), falhando em replicar gargalos crônicos reais (como na Ponte Newton Navarro). Além disso, desconsiderou o relevo/clima na caminhada humana, a presença de semáforos e conversões proibidas.

12 - Como o modelo poderia ser aproximado de um aplicativo real de mobilidade?

Integrando o sistema com APIs de trânsito em tempo real (como Google Maps ou Waze), adicionando filtros de segurança urbana (evitar pontos perigosos à noite), adicionando restrições de acessibilidade para a caminhada e respeitando todas as regras de trânsito locais.